<a href="https://colab.research.google.com/github/kora-ai-lab/h-azr/blob/main/notebooks/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KAIL — Notebook 00: Environment Setup

> **Kora AI Lab** | Run this once before any other notebook.

This notebook:
1. Checks your GPU and VRAM
2. Installs all dependencies
3. Logs into HuggingFace (needed to download models)
4. Mounts Google Drive (to persist checkpoints between sessions)
5. Runs a quick smoke test

**Runtime:** ~5 minutes

In [1]:
# @title Step 1 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime > Change runtime type > T4 GPU"

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1e9

print(f"GPU:   {gpu.name}")
print(f"VRAM:  {vram_gb:.1f} GB")
print(f"CUDA:  {torch.version.cuda}")

if vram_gb < 14:
    print("WARNING: Less than 14GB VRAM. Some notebooks may OOM. Try Kaggle (2x T4 = 32GB) instead.")
else:
    print("VRAM OK for all notebooks.")

Sat Mar 21 16:44:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# @title Step 2 — Install dependencies
# This takes ~3-4 minutes
!pip install -q \
  torch==2.3.0 \
  transformers==4.44.0 \
  datasets==2.20.0 \
  accelerate==0.33.0 \
  evaluate==0.4.2 \
  trl==0.9.6 \
  peft==0.12.0 \
  bitsandbytes==0.43.3 \
  wandb==0.17.5 \
  scikit-learn==1.5.0 \
  huggingface_hub==0.24.5 \
  safetensors==0.4.3 \
  sentencepiece==0.2.0

print("Installation complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [5]:
# @title Step 3 — HuggingFace login (needed to download Qwen)
from huggingface_hub import login, whoami

# Option A: paste your token here (get it from huggingface.co/settings/tokens)
HF_TOKEN = ""  # paste your token between the quotes

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    # Option B: interactive login
    login()

user = whoami()
print(f"Logged in as: {user['name']}")

Logged in as: kheirli


In [9]:
# @title Step 4 — Mount Google Drive (to save checkpoints between sessions)
# Checkpoints saved here survive Colab session resets
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/KAIL/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print("All notebooks will save here automatically.")

Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/KAIL/checkpoints
All notebooks will save here automatically.


In [10]:
# @title Step 5 — Smoke test: load model tokenizer
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
print(f"Testing download of {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

test = tokenizer("Hello KAIL", return_tensors="pt")
print(f"Tokenizer OK. Vocab size: {tokenizer.vocab_size:,}")
print("\nSetup complete. You can now run the numbered notebooks in order:")
print("  01_baseline_eval.ipynb    <- Run first (Scenario A)")
print("  02_h_neuron_probe.ipynb   <- Identify H-Neurons")
print("  03_spin_warmup.ipynb      <- Stage 1 SPIN training")
print("  04_h_azr_training.ipynb   <- Stage 3 H-AZR (our contribution)")

Testing download of Qwen/Qwen2.5-Coder-1.5B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer OK. Vocab size: 151,643

Setup complete. You can now run the numbered notebooks in order:
  01_baseline_eval.ipynb    <- Run first (Scenario A)
  02_h_neuron_probe.ipynb   <- Identify H-Neurons
  03_spin_warmup.ipynb      <- Stage 1 SPIN training
  04_h_azr_training.ipynb   <- Stage 3 H-AZR (our contribution)
